### Keyspace Definition

Keyspace definition is the process of specifying the basic data dimensions that define a simulation environment. 

Keyspaces are typically first defined in an abstract manner and then instantiated as part of model construction. Formally, a model's keyspace consists of a three-level hierarchy of symbolic keys. These keys collectively define basic terms for indexing simulation data. Within a model's keyspace, a key may be classified as a *term*, *sort*, or *family* according to its level (from lowest to highest). A term names an individual data dimension (e.g., an individual feature, parameter, etc.), a sort groups terms that are alike in content (e.g., color terms, shape terms, etc.), and a family groups sorts that play like roles within a simulation (e.g., data, parameters, etc.). 

For the domain of the WCST, symbols may be defined to represent various figural properties such as the color, shape, and number of figures displayed on a card. Furthermore, some additional terms, representing different data roles (inputs, outputs, targets, etc.) may also be needed.  

In [ ]:
"""Data hierarchy for the WCST domain."""

from pyClarion import Atom # Represents an atomic term
from pyClarion import Atoms # Represents a sort grouping atomic terms
from pyClarion.knowledge import Buses, Bus, BusFamily, DataFamily, Root
import random

#A card can have the shapes circle, square, or triangle, each having the same colors which can be red, green, or blue, and there can be 1-3 of the shapes in a single card.

class Color(Atoms): 
    """A sort for atomic color terms."""
    red: Atom
    grn: Atom
    blu: Atom


class Shape(Atoms): 
    """A sort for atomic shape terms."""
    circ: Atom
    squr: Atom
    tria: Atom


class Number(Atoms):
    """A sort for atomic number terms."""
    one: Atom
    two: Atom
    three: Atom


class Main(Buses):
    """A sort for main data buses."""
    input: Bus
    output: Bus
    target: Bus


class WCSTBuses(BusFamily):
    """A family for all bus sorts."""
    main: Main


class WCSTData(DataFamily):
    """A family for all data sorts."""
    color: Color
    shape: Shape
    number: Number
    


class WCSTRoot(Root):
    """The root of the model keyspace."""
    b: WCSTBuses
    d: WCSTData





### Model Construction

Model construction is the process of initializing and assembling a collection of `pyClarion` component processes that implement the functionality of a desired model. 

The `pyClarion` library defines component processes implementing the main elements of the Clarion cognitive architecture, and it allows models to be constructed in a modular and compositional fashion.

For the present purposes, model construction requires the `Agent`, `Input`, and `ChunkStore` processes and the `WCSTData` keyspace.

In [53]:
"""Agent construction for bottom-up activation model."""

from pyClarion import Agent # Initialzes keyspaces specific to an agent
from pyClarion import Input # Receives activations from external sources
from pyClarion import ChunkStore # Maintains a collection of Clarion chunks
from pyClarion import BottomUp
from pyClarion.knowledge import DVPairs


class BottomUpAgent[R: Root, D: DVPairs](Agent):
    root: R
    ipt: Input
    chunks: ChunkStore[D]
    bu: BottomUp[D]

    def __init__(self, name: str, root: R, f: DataFamily, d: D) -> None:
        
        super().__init__(name, root)

        # Convenient handle for keyspace root.
        self.root = root
        
        # Any process objects initialized within this block will automatically 
        # be added to the agent's system.
        with self:
            # Create an input process to pass in feature activations. The tuple 
            # `(b, d)`indicates that this process will receive inputs in the 
            # form of dimension-value pairs constructed by pairing symbols from 
            # the families 'b'  and 'd'.
            self.ipt = Input(f"{name}.ipt", d)

            # Create a chunk store using the family 'd' to house chunk symbols 
            # (arg 'c')
            self.chunks = ChunkStore(f"{name}.chunks", c=f, d=d)

            # Create a bottom up activation process dependent on chunk store.
            self.bu = self.ipt >> self.chunks.bottom_up(f"{name}.bu")

# Initialize basic data symbols for current simulation
# During initialization, WCSTData() will automatically generate symbols 
# for all terms annotated with a sort or term type.
root = WCSTRoot()

# Create a new bottom-up demo agent and populate its keyspace with data symbols.
agent = BottomUpAgent("Benny", root, root.d, (root.b.main, root.d))

# List all processes associated with the current model
agent.system.procs

[<BottomUpAgent 'Benny' at 0x10dcb0440>,
 <Input 'Benny.ipt' at 0x10dbfb9b0>,
 <ChunkStore 'Benny.chunks' at 0x10dbfbac0>,
 <BottomUp 'Benny.bu' at 0x10dbfbce0>]

### Knowledge Initialization

Knowledge initialization is the process of defining any prior knowledge available to a model. 

Typically, this involves the specification of user-defined explicit knowledge such as chunks and/or rules using `pyClarion`'s knowledge representation tools. It may also, however, include populating the model with other prior knowledge, such as pretrained neural network weights. 

For the present example, knowledge initialization involves defining some chunks for which to calculate bottom-up activations. Three chunks are defined below, representing three distinct card faces.

In [54]:
"""Chunks for bottom-up activation model"""

main = agent.root.b.main
color = agent.root.d.color   
shape = agent.root.d.shape  
number = agent.root.d.number 

# This defines the cards available to the model. The sort of the cards defines all the possible categorization rules (color, number, or shape)

chunk_defs = [

    "one_blue_triangle" ^ 
    + main.input ** color.blu
    + main.input ** shape.tria
    + main.input ** number.one,

    "two_red_circles" ^
    + main.input ** color.red
    + main.input ** shape.circ
    + main.input ** number.two,

    "three_green_squares" ^
    + main.input ** color.grn
    + main.input ** shape.squr
    + main.input ** number.three,
]

agent.system.schedule(agent.chunks.encode(*chunk_defs))

### Event Processing

Event processing is the process of getting a model to generate and process simulation events. 

Events have four primary effects on the state of a simulation: They may (i) update numerical data in process sites, (ii) update the simulation's keyspace, (iii) cause processes to schedule further events, or (iv) advance the simulation clock. Events do not actively perform computations, thus updates may only depend on the state of a model at the time that an event is scheduled.

Scheduled events are maintained in an event queue, which ensures that they are processed in due time. The preceding step has already generated one event which, when processed, will have all of the effects listed above.  


In [55]:
# List all currently scheduled events. Technically, the event queue is 
# implemented as a heap, so events are not guaranteed to be listed in order.
agent.system.queue

[<Event source=Benny.chunks.encode time=datetime.timedelta(0) at 0x10dc1fdc0>]

To facilitate tracking events and other relevant data during the course of a simulation, event logs may be generated using Python's `logging` module.

In [56]:
"""Initialize event logging for simulation"""

import logging
import sys

logger = logging.getLogger("pyClarion.events.system")
logger.setLevel(logging.DEBUG)
logger.handlers.clear()
logger.addHandler(logging.StreamHandler(sys.stdout))

The present simulation may be completed simply by sending some inputs into the bottom level of the model.

In [57]:
"""Run simulation"""

# Schedule an event to update the input to the model with the following data.
# Positives get +1.0 activation, negatives get -1.0 activation. 
agent.system.schedule(
    agent.ipt.send(    
        + main.input ** color.blu
        + main.input ** shape.tria
        + main.input ** number.one
        - main.input ** number.two
    ) 
)

# The following lines process all events in the current queue.
print("Event summaries generated by logger.\n")
for event in agent.run(): # Process and yield the next event.
    continue
    # Can respond to event here (e.g., record data, stop simulation, 
    # communicate with external resources etc.)

Event summaries generated by logger.

event 0x0000 00:00:00.00 096 0 Benny.chunks.encode
    Added the following new chunk(s)
    chunk d:Benny.chunks:one_blue_triangle
        + main.input ** color.blu
        + main.input ** shape.tria
        + main.input ** number.one
    chunk d:Benny.chunks:two_red_circles
        + main.input ** color.red
        + main.input ** shape.circ
        + main.input ** number.two
    chunk d:Benny.chunks:three_green_squares
        + main.input ** color.grn
        + main.input ** shape.squr
        + main.input ** number.three
event 0x0000 00:00:00.00 096 2 Benny.chunks.encode_weights
event 0x0000 00:00:00.00 064 1 Benny.ipt.send
event 0x0000 00:00:00.00 064 3 Benny.bu.forward


## Results

The result of computing bottom-up activations from the given input may be obtained by accessing the main output site of the bottom-up activation process `chunks.bu`. This presents an occasion for a glimpse into the internal representations of a `pyClarion` model.

In [21]:
# The bottom-up output is stored in the data site called bu.main. The 
# current data at a site is accessed via subscripting at index 0. Some sites 
# retain lagged data, which may be accessed by subscripting with the 
# corresponding lag value.
data = agent.bu.main[0]

# Print the outcome of the bottom-up activation process.
# The first line of the result contains some general information, each 
# subsequent line lists key-value associations which, in this case, represent 
# bottom-up activations for each chunk. Consult the event logs for information 
# on the chunk identifiers.
print(data)

NumDict 'd:Benny.chunks:?' c=0.0
    d:Benny.chunks:one_blue_triangle 0.75
    d:Benny.chunks:two_red_circles   -0.25
